### Sentiment Analysis
Sentiment Analysis is an NLP task that identifies the emotional tone or opinion expressed in text.

It classifies text into categories such as:

Positive

Negative

Neutral

(Sometimes) emotions like joy, anger, sadness

How it works:

Convert text into numerical features (e.g., TF-IDF, word embeddings, transformer embeddings).

Train a model to predict sentiment.

Common Approaches:

Traditional ML: Logistic Regression, Naive Bayes, SVM

Deep Learning: LSTM, CNN

Transformers: BERT, RoBERTa


### Logistic Regression + TF-IDF

In [1]:
!pip install datasets -q

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split


In [2]:
dataset = load_dataset("imdb")


train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)


In [4]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_labels)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(test_labels, preds))
print(classification_report(test_labels, preds))


Accuracy: 0.88224
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



## Toxic Comment Detector
Dataset: Jigsaw Toxic Comment Classification Challenge

Model: Logistic Regression

Vectorizer: TF-IDF

Metrics: Accuracy + Precision + Recall + F1

In [5]:
!pip install kaggle --quiet


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [7]:
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge
!unzip jigsaw-toxic-comment-classification-challenge.zip


Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 434, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.config/kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/
unzip:  cannot find or open jigsaw-toxic-comment-classification-challenge.zip, jigsaw-toxic-comment-classification-challenge.zip.zip or jigsaw-toxic-comment-classification-challenge.zip.ZIP.


In [8]:
df = pd.read_csv("train.csv")
df.head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [9]:
df = df[["comment_text", "toxic"]]
df = df.dropna()


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df["comment_text"],
    df["toxic"],
    test_size=0.2,
    random_state=42
)


In [11]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [12]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)


LogisticRegression(max_iter=1000)

In [13]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy:  0.956384145386182
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28859
           1       0.90      0.61      0.73      3056

    accuracy                           0.96     31915
   macro avg       0.93      0.80      0.85     31915
weighted avg       0.95      0.96      0.95     31915



In [14]:
def predict_toxic(comment):
    vec = tfidf.transform([comment])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1]

    print("Toxic" if pred == 1 else "Not Toxic")
    print("Confidence:", round(prob, 3))

predict_toxic("You are stupid and useless")
predict_toxic("Have a nice day!")


Toxic
Confidence: 0.997
Not Toxic
Confidence: 0.077


### LSTM (Long Short-Term Memory)
LSTM (Long Short-Term Memory) is a type of recurrent neural network (RNN) used in sentiment analysis because it can understand the order and context of words in a sentence.

Unlike traditional machine learning models that treat words independently, LSTM processes text sequentially, remembering important information and forgetting irrelevant parts. This allows it to correctly interpret phrases like:

“not good”

“I don’t hate this”

where word order changes the sentiment.

In [15]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [16]:
vocab_size = 10000  # Use top 10,000 words

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


Training samples: 25000
Test samples: 25000


In [17]:
max_length = 200

X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)


In [18]:
model = Sequential()

model.add(Embedding(input_dim=vocab_size,
    output_dim=64,
    input_length=max_length ))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=["accuracy"])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)


Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 148ms/step - accuracy: 0.7014 - loss: 0.5485 - val_accuracy: 0.8714 - val_loss: 0.3098
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 150ms/step - accuracy: 0.9039 - loss: 0.2560 - val_accuracy: 0.8652 - val_loss: 0.3461
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 44s 139ms/step - accuracy: 0.9323 - loss: 0.1833 - val_accuracy: 0.8606 - val_loss: 0.3530


In [20]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - accuracy: 0.8560 - loss: 0.3770
Test Accuracy: 0.8573600053787231


In [21]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_length = 200


word_index = tf.keras.datasets.imdb.get_word_index()

def encode_review(text):
    tokens = text.lower().split()
    encoded = []
    for word in tokens:
        if word in word_index and word_index[word] < vocab_size:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # unknown word
    return pad_sequences([encoded], maxlen=max_length)

def predict_sentiment(text):
    encoded = encode_review(text)
    prediction = model.predict(encoded)[0][0]

    print("Review:", text)
    print("Sentiment:", "Positive" if prediction > 0.5 else "Negative")
    print("Confidence:", round(float(prediction), 3))

predict_sentiment("This movie was amazing and wonderful")
predict_sentiment("This movie was awesome")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
Review: This movie was amazing and wonderful
Sentiment: Positive
Confidence: 0.977
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Review: This movie was awesome
Sentiment: Positive
Confidence: 0.781


In [22]:
!pip install tensorflow scikit-learn matplotlib seaborn --quiet


In [23]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import regularizers

from sklearn.metrics import classification_report, confusion_matrix, f1_score


In [24]:
max_vocab = 20000

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=max_vocab
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


Training samples: 25000
Test samples: 25000


In [25]:
max_len = 250

X_train = pad_sequences(X_train, maxlen=max_len, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=max_len, padding="post", truncating="post")


### Build Advanced Bidirectional LSTM Model

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras import regularizers
import tensorflow as tf

model = Sequential()

model.add(Embedding(input_dim=max_vocab,
                    output_dim=128,
                    input_length=max_len))

# FIXED HERE
model.add(Bidirectional(LSTM(64)))

model.add(Dropout(0.5))

model.add(Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(0.001)
))

model.add(Dropout(0.5))

model.add(Dense(1, activation="sigmoid"))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_lstm_model.h5",
    monitor="val_loss",
    save_best_only=True
)


In [28]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=64,
    callbacks=[early_stop, checkpoint]
)


Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 553ms/step - accuracy: 0.5960 - loss: 0.6895

313/313 ━━━━━━━━━━━━━━━━━━━━ 194s 590ms/step - accuracy: 0.5963 - loss: 0.6892 - val_accuracy: 0.8228 - val_loss: 0.4240
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 189s 555ms/step - accuracy: 0.8828 - loss: 0.3236 - val_accuracy: 0.8180 - val_loss: 0.4337
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 517ms/step - accuracy: 0.9307 - loss: 0.2139

313/313 ━━━━━━━━━━━━━━━━━━━━ 173s 553ms/step - accuracy: 0.9307 - loss: 0.2139 - val_accuracy: 0.8674 - val_loss: 0.3874
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 172s 550ms/step - accuracy: 0.9583 - loss: 0.1359 - val_accuracy: 0.8312 - val_loss: 0.4454
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 210s 577ms/step - accuracy: 0.9631 - loss: 0.1160 - val_accuracy: 0.8664 - val_loss: 0.4680


In [29]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


782/782 ━━━━━━━━━━━━━━━━━━━━ 54s 69ms/step - accuracy: 0.8323 - loss: 0.4677
Test Accuracy: 0.8303999900817871


In [30]:
from tensorflow import keras
word_index = keras.datasets.imdb.get_word_index()
max_features = 10000
maxlen = 200


def encode_review(text):
    tokens = text.lower().split()
    encoded = []
    for word in tokens:
        if word in word_index and word_index[word] < max_features:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # unknown
    return pad_sequences([encoded], maxlen=maxlen)

def predict_sentiment(text):
    encoded = encode_review(text)
    pred = model.predict(encoded)[0][0]

    print("Text:", text)
    print("Sentiment:", "Positive" if pred > 0.5 else "Negative")
    print("Confidence:", round(float(pred), 3))

predict_sentiment("This movie was amazing and wonderful")
predict_sentiment("This movie was terrible and boring")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step
Text: This movie was amazing and wonderful
Sentiment: Negative
Confidence: 0.5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Text: This movie was terrible and boring
Sentiment: Positive
Confidence: 0.509
